# Attention Gated Recurrent Grad-CAM pour la vidéo — Entraînement et évaluation sur UCF101

Ce notebook contient l'implémentation complète de l'approche **Attention-Gated Recurrent Grad-CAM** utilisée comme modèle principal dans ce travail.
 
L'objectif est de construire un modèle vidéo capable de produire simultanément :
 
 - une **prédiction** de l'action vidéo ;
 - une explication spatiale indiquant **où** le modèle regarde ;
 - une explication temporelle indiquant **quand** les frames deviennent importantes ;
 - une explication dynamique indiquant **comment** le mouvement contribue à la décision.
 
Ce notebook est volontairement limité à l'entraînement, l'inférence, la visualisation et l'évaluation quantitative du modèle Attention-Gated Grad-CAM.  


 
## Plan du notebook
 
 1. Configuration expérimentale et environnement
 2. Chargement et préparation du dataset UCF101
 3. Fonctions utilitaires pour la vidéo et la dynamique
 4. Architecture du modèle Attention-Gated Grad-CAM
 5. Contraintes XAI et fonction de perte mixte
 6. Génération des explications Grad-CAM spatio-temporelles
 7. Métriques d'évaluation
 8. Entraînement, validation et checkpoints
 9. Explication qualitative d'une vidéo
 10. Exécution expérimentale
 11. Evaluation



## 1. Configuration expérimentale et environnement

Cette section définit les bibliothèques, les chemins du dataset UCF101, les paramètres d'entraînement et les hyperparamètres XAI.

Le dataset attendu est organisé comme suit :
 
```text
/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/
    train/<classe>/*.avi
    val/<classe>/*.avi
    test/<classe>/*.avi
```

Les paramètres importants sont :

- `clip_len` : nombre de frames utilisées par vidéo ;
- `frame_size` : taille spatiale des frames ;
- `backbone_name` : backbone vidéo utilisé, ici S3D par défaut ;
- coefficients `lambda_*` : poids des contraintes d'explicabilité.

In [1]:
# Imports des biblios

import os
import cv2
import math
import json
import random
import shutil
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import s3d, S3D_Weights, r3d_18, R3D_18_Weights

from IPython.display import Video, display, HTML

try:
    from tqdm import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x


In [2]:
# Configuration

@dataclass
class CFG:
    dataset_root: str = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition"
    backbone_name: str = "s3d"  # "s3d" est I3D-like
    pretrained: bool = True
    num_classes: int = 101

    clip_len: int = 32
    frame_size: int = 224
    batch_size: int = 4
    num_workers: int = 2
    epochs: int = 10
    lr: float = 1e-4
    weight_decay: float = 1e-4

    # Temporal gate. Default is LSTM-attention because it preserved accuracy better in our experiments.
    # Set temporal_module="transformer" to reproduce the Transformer ablation.
    temporal_module: str = "lstm_attention"
    lstm_hidden_dim: int = 256
    lstm_bidirectional: bool = True
    transformer_layers: int = 1
    transformer_heads: int = 8
    transformer_dropout: float = 0.1
    temporal_hidden_dim: int = 512

    # Progressive XAI training schedule.
    # First epochs learn classification; then XAI constraints are activated gradually.
    xai_warmup_epochs: int = 2
    xai_ramp_epochs: int = 2

    # Balanced explanation constraints. Values are intentionally softer than V4 to recover accuracy.
    lambda_spatial_sparse: float = 0.02
    lambda_spatial_tv: float = 0.02
    lambda_temporal_entropy: float = 0.03
    lambda_temporal_smooth: float = 0.03
    lambda_temporal_coverage: float = 0.05
    lambda_motion_align: float = 0.15
    lambda_spatial_focus: float = 0.05
    lambda_input_fidelity: float = 0.20
    lambda_strong_causal: float = 0.10
    lambda_sufficiency: float = 0.10

    target_temporal_coverage: float = 0.35
    fidelity_margin: float = 0.35
    topk_temporal_ratio: float = 0.35
    topk_spatial_ratio: float = 0.25
    deletion_fill: str = "mean"  # "zero" or "mean"

    # Evaluation fidelity settings.
    # These parameters are used only for post-hoc evaluation, not for training.
    eval_steps: int = 8
    eval_spatiotemporal_top_ratio: float = 0.35
    eval_spatial_dilation: int = 11
    eval_deletion_fill: str = "blur"  # "blur", "mean", or "zero"

    # Optical flow settings
    use_optical_flow: bool = True
    flow_size: int = 112
    flow_topk_ratio: float = 0.30

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    output_dir: str = "./xai_outputs_attention_gated_final"


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def split_dir(cfg: CFG, split: str) -> str:
    return os.path.join(cfg.dataset_root, split)


## 2. Chargement et préparation du dataset UCF101
 
Cette section définit un dataset vidéo de type `VideoFolderDataset`, compatible avec l'organisation Kaggle de UCF101.
 
Chaque vidéo est lue, échantillonnée temporellement, redimensionnée, normalisée puis retournée sous la forme :
 
$$X \in \mathbb{R}^{C \times T \times H \times W}$$

avec :
 
 - `C` : canaux RGB ;
 - `T` : nombre de frames ;
 - `H, W` : dimensions spatiales.


In [3]:
# Dataset UCF101 Folder Loader

class UCF101VideoFolderDataset(Dataset):
    """
    Folder layout:
        root_dir/class_name/video.avi
    """
    def __init__(
        self,
        root_dir: str,
        clip_len: int = 32,
        frame_size: int = 224,
        train: bool = False,
        class_to_idx: Optional[Dict[str, int]] = None,
        max_videos_per_class: Optional[int] = None,
    ):
        self.root_dir = root_dir
        self.clip_len = clip_len
        self.frame_size = frame_size
        self.train = train

        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = class_to_idx if class_to_idx is not None else {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

        exts = (".avi", ".mp4", ".mov", ".mkv")
        self.samples = []
        for class_name in classes:
            class_path = os.path.join(root_dir, class_name)
            videos = sorted([f for f in os.listdir(class_path) if f.lower().endswith(exts)])
            if max_videos_per_class is not None:
                videos = videos[:max_videos_per_class]
            if class_name not in self.class_to_idx:
                continue
            for f in videos:
                self.samples.append((os.path.join(class_path, f), self.class_to_idx[class_name], class_name))

        self.normalize = transforms.Normalize(
            mean=[0.43216, 0.394666, 0.37645],
            std=[0.22803, 0.22145, 0.216989],
        )

    def __len__(self):
        return len(self.samples)

    def _read_video(self, path: str) -> List[np.ndarray]:
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        return frames

    def _sample_indices(self, n: int) -> np.ndarray:
        if n <= 0:
            return np.zeros(self.clip_len, dtype=np.int64)
        if n >= self.clip_len:
            if self.train:
                start = random.randint(0, max(0, n - self.clip_len))
                return np.arange(start, start + self.clip_len)
            return np.linspace(0, n - 1, self.clip_len).astype(np.int64)
        return np.linspace(0, n - 1, self.clip_len).astype(np.int64)

    def _preprocess(self, frames: List[np.ndarray]) -> Tuple[torch.Tensor, torch.Tensor]:
        idx = self._sample_indices(len(frames))
        sampled = []
        for i in idx:
            if len(frames) == 0:
                fr = np.zeros((self.frame_size, self.frame_size, 3), dtype=np.uint8)
            else:
                fr = frames[int(i)]
            fr = cv2.resize(fr, (self.frame_size, self.frame_size))
            if self.train and random.random() < 0.5:
                fr = cv2.flip(fr, 1)
            sampled.append(fr)

        arr = np.stack(sampled).astype(np.float32) / 255.0  # T,H,W,C

        raw = torch.from_numpy(arr).permute(3, 0, 1, 2).float()  # C,T,H,W

        x = raw.clone()
        for t in range(x.shape[1]):
            x[:, t] = self.normalize(x[:, t])

        return x, raw

    def __getitem__(self, idx: int):
        path, y, class_name = self.samples[idx]
        frames = self._read_video(path)
        x, raw = self._preprocess(frames)
        return {
            "video": x,
            "raw_video": raw,
            "label": torch.tensor(y, dtype=torch.long),
            "path": path,
            "class_name": class_name,
        }


def build_datasets(cfg: CFG):
    train_ds = UCF101VideoFolderDataset(split_dir(cfg, "train"), cfg.clip_len, cfg.frame_size, train=True)
    val_ds = UCF101VideoFolderDataset(split_dir(cfg, "val"), cfg.clip_len, cfg.frame_size, train=False, class_to_idx=train_ds.class_to_idx)
    test_ds = UCF101VideoFolderDataset(split_dir(cfg, "test"), cfg.clip_len, cfg.frame_size, train=False, class_to_idx=train_ds.class_to_idx)
    print(f"Train videos: {len(train_ds)} | Val videos: {len(val_ds)} | Test videos: {len(test_ds)}")
    print(f"Classes: {len(train_ds.class_to_idx)}")
    return train_ds, val_ds, test_ds


def build_loaders(cfg: CFG):
    train_ds, val_ds, test_ds = build_datasets(cfg)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds



## 3. Fonctions utiles pour la vidéo et la dynamique

In [4]:
# Video denormalization and optical-flow motion targets

def denormalize_video(x: torch.Tensor) -> np.ndarray:
    """
    x: [C,T,H,W] normalized or raw.
    returns uint8 [T,H,W,C]
    """
    x = x.detach().cpu()
    if x.min() < 0:
        mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
        std = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)
        x = x * std + mean
    x = x.clamp(0, 1)
    return (x.permute(1, 2, 3, 0).numpy() * 255).astype(np.uint8)


@torch.no_grad()
def compute_optical_flow_targets(
    video: torch.Tensor,
    out_hw: Optional[Tuple[int, int]] = None,
    flow_size: int = 112,
    topk_ratio: float = 0.30,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Computes optical-flow magnitude pseudo-targets.

    input video: [B,C,T,H,W], normalized or raw.
    returns:
        temporal_flow: [B,T] normalized flow energy.
        spatial_flow: [B,1,T,Hout,Wout] normalized map in [0,1].
        binary_focus: [B,1,T,Hout,Wout] top-k pseudo object/motion region.
    """
    B, C, T, H, W = video.shape
    Hout, Wout = out_hw if out_hw is not None else (H, W)

    temporal_list = []
    spatial_list = []

    for b in range(B):
        frames = denormalize_video(video[b])
        gray_seq = []
        for fr in frames:
            gray = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY)
            gray = cv2.resize(gray, (flow_size, flow_size))
            gray_seq.append(gray)

        mags = []
        if T <= 1:
            mags = [np.zeros((flow_size, flow_size), dtype=np.float32)]
        else:
            first_mag = None
            for t in range(T - 1):
                prev = gray_seq[t]
                nxt = gray_seq[t + 1]
                flow = cv2.calcOpticalFlowFarneback(
                    prev, nxt, None,
                    pyr_scale=0.5, levels=3, winsize=15,
                    iterations=3, poly_n=5, poly_sigma=1.2, flags=0
                )
                mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2).astype(np.float32)
                if first_mag is None:
                    first_mag = mag
                mags.append(mag)
            mags = [first_mag] + mags

        mags = np.stack(mags)  # T,h,w
        maps = []
        for t in range(T):
            m = mags[t]
            m = m / (m.max() + 1e-8)
            m = cv2.resize(m, (Wout, Hout))
            maps.append(m)
        maps = np.stack(maps).astype(np.float32)  # T,Hout,Wout

        temporal = maps.mean(axis=(1, 2))
        temporal = temporal / (temporal.sum() + 1e-8)

        temporal_list.append(temporal)
        spatial_list.append(maps)

    temporal_flow = torch.tensor(np.stack(temporal_list), dtype=torch.float32, device=video.device)
    spatial_flow = torch.tensor(np.stack(spatial_list), dtype=torch.float32, device=video.device).unsqueeze(1)

    flat = spatial_flow.flatten(2)
    k = max(1, int(flat.shape[-1] * topk_ratio))
    thresh = torch.topk(flat, k=k, dim=-1).values[..., -1].view(B, 1, 1, 1, 1)
    binary_focus = (spatial_flow >= thresh).float()

    return temporal_flow, spatial_flow, binary_focus


@torch.no_grad()
def compute_frame_diff_targets(
    video: torch.Tensor,
    out_hw: Optional[Tuple[int, int]] = None,
    topk_ratio: float = 0.30,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Fast fallback if optical flow is too slow.
    """
    B, C, T, H, W = video.shape
    Hout, Wout = out_hw if out_hw is not None else (H, W)
    raw = video.detach()
    if raw.min() < 0:
        mean = torch.tensor([0.43216, 0.394666, 0.37645], device=video.device).view(1, 3, 1, 1, 1)
        std = torch.tensor([0.22803, 0.22145, 0.216989], device=video.device).view(1, 3, 1, 1, 1)
        raw = (raw * std + mean).clamp(0, 1)
    diff = torch.abs(raw[:, :, 1:] - raw[:, :, :-1]).mean(dim=1, keepdim=True)
    first = diff[:, :, :1]
    maps = torch.cat([first, diff], dim=2)
    maps = F.interpolate(maps, size=(T, Hout, Wout), mode="trilinear", align_corners=False)
    maps = maps / (maps.amax(dim=(2, 3, 4), keepdim=True) + 1e-8)
    temporal = maps.mean(dim=(1, 3, 4))
    temporal = temporal / (temporal.sum(dim=1, keepdim=True) + 1e-8)

    flat = maps.flatten(2)
    k = max(1, int(flat.shape[-1] * topk_ratio))
    thresh = torch.topk(flat, k=k, dim=-1).values[..., -1].view(B, 1, 1, 1, 1)
    binary_focus = (maps >= thresh).float()
    return temporal, maps, binary_focus


from typing import Dict, Any, Optional
def describe_dynamics_from_flow(
    temporal_attention: torch.Tensor,
    temporal_flow: torch.Tensor,
    heatmap: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """
    Génère une explication dynamique pondérée par :
        - l'attention temporelle du modèle ;
        - l'intensité du mouvement (optical flow) ;
        - les régions Grad-CAM importantes.

    Retour :
        dictionnaire compact pour affichage explicatif.
    """

    att = temporal_attention.detach().cpu().float()
    flow = temporal_flow.detach().cpu().float()

    if att.numel() != flow.numel():
        att = F.interpolate(
            att.view(1, 1, -1),
            size=flow.numel(),
            mode="linear",
            align_corners=False
        ).view(-1)

    att_np = att.numpy()
    flow_np = flow.numpy()

    important = np.where(att_np >= np.percentile(att_np, 75))[0]

    if len(important) == 0:
        important = np.array([int(np.argmax(att_np))])

    peak = int(np.argmax(att_np))

    # ============================================================
    # Intensité dynamique
    # ============================================================

    motion_score = float(flow_np[important].mean())

    if motion_score >= 0.065:
        motion_level = "fort"
    elif motion_score >= 0.035:
        motion_level = "moyen"
    else:
        motion_level = "faible"

    # ============================================================
    # Variation / accélération
    # ============================================================

    accel = np.abs(np.diff(flow_np))

    accel_score = float(accel[important[:-1]].mean()) if len(important) > 1 else float(accel.mean())

    if accel_score >= 0.020:
        accel_level = "élevée"
    elif accel_score >= 0.010:
        accel_level = "modérée"
    else:
        accel_level = "faible"

    # ============================================================
    # Continuité temporelle
    # ============================================================

    important_binary = np.zeros_like(att_np)
    important_binary[important] = 1

    switches = np.abs(np.diff(important_binary)).sum()

    if switches <= 2:
        continuity = "stable"
    elif switches <= 5:
        continuity = "modérée"
    else:
        continuity = "intermittente"

    # ============================================================
    # Localisation dynamique via GradCAM
    # ============================================================

    localization_ratio = None
    localization_label = "indéterminé"

    if heatmap is not None:
        try:
            hm = np.clip(heatmap, 0, 1)

            spatial_focus = []
            for t in important:
                h = hm[t]
                active = h >= np.percentile(h, 75)

                if active.sum() == 0:
                    continue

                ratio = active.mean()
                spatial_focus.append(ratio)

            if len(spatial_focus) > 0:
                localization_ratio = 1.0 - float(np.mean(spatial_focus))

                if localization_ratio >= 0.70:
                    localization_label = "localisé"
                elif localization_ratio >= 0.45:
                    localization_label = "mixte"
                else:
                    localization_label = "global"

        except Exception:
            pass

    # ============================================================
    # Direction dominante approximative
    # ============================================================

    diffs = np.diff(flow_np)

    if len(diffs) > 0:
        signed_motion = diffs.mean()

        if signed_motion > 0.002:
            direction = "accélération progressive"
        elif signed_motion < -0.002:
            direction = "ralentissement"
        else:
            direction = "mouvement stable"
    else:
        direction = "stable"

    return {
        "segment": f"frames {int(important.min())} à {int(important.max())}",
        "motion_intensity_label": motion_level,
        "motion_intensity_value": round(motion_score, 4),
        "localization_label": localization_label,
        "localization_value": None if localization_ratio is None else round(localization_ratio * 100, 1),
        "variation_label": accel_level,
        "variation_value": round(accel_score, 4),
        "continuity": continuity,
        "direction": direction,
        "peak_frame": peak,
    }


## 4. Architecture du modèle Attention-Gated Grad-CAM
 
L'architecture suit une logique explicable par design :

1. Un backbone vidéo S3D extrait des caractéristiques spatio-temporelles.
2. Une porte spatiale apprend les régions importantes de la vidéo.
3. Un module temporel LSTM-attention apprend les frames importantes.
4. Le classifieur produit la prédiction finale.
5. Les cartes Grad-CAM sont fusionnées avec les portes spatiales et temporelles.

Cette structure permet de couvrir les trois axes d'explication :

- **où** : régions spatiales importantes ;
- **quand** : frames ou segments importants ;
- **comment** : dynamique et mouvement associés à la décision.


In [5]:
# Model: Spatial Gate + Temporal Transformer Gate

class PositionalEncoding(nn.Module):
    def __init__(self, dim: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.shape[1]]


class SpatialGate3D(nn.Module):
    """
    Learns WHERE to look.
    Input: [B,C,T,H,W]
    Output: [B,1,T,H,W]
    """
    def __init__(self, in_channels: int):
        super().__init__()
        mid = max(in_channels // 8, 16)
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, mid, kernel_size=1),
            nn.BatchNorm3d(mid),
            nn.ReLU(inplace=True),
            nn.Conv3d(mid, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)



class TemporalLSTMAttentionGate(nn.Module):
    """
    Learns WHEN to look using a recurrent temporal memory followed by attention.
    This is the default final variant because it keeps better classification accuracy
    while still providing explicit temporal importance.
    """
    def __init__(self, in_channels: int, hidden_dim: int = 256, bidirectional: bool = True, dropout: float = 0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=in_channels,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.att_head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim // 2, 1),
        )
        self.context_proj = nn.Sequential(nn.LayerNorm(out_dim), nn.Linear(out_dim, out_dim), nn.Tanh())
        self.out_dim = out_dim

    def forward(self, frame_features: torch.Tensor):
        seq, _ = self.lstm(frame_features)
        logits = self.att_head(seq).squeeze(-1)
        beta = torch.sigmoid(logits)
        alpha = beta / (beta.sum(dim=1, keepdim=True) + 1e-8)
        context = torch.sum(seq * alpha.unsqueeze(-1), dim=1)
        context = self.context_proj(context)
        return context, alpha, beta, seq

class TemporalTransformerGate(nn.Module):
    """
    Learns WHEN to look.
    Replaces LSTM by Transformer Encoder.
    Uses sigmoid beta for coverage, and normalized alpha for context aggregation.
    """
    def __init__(self, in_channels: int, model_dim: int = 512, n_heads: int = 8, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(in_channels, model_dim)
        self.pos = PositionalEncoding(model_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=n_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.att_head = nn.Sequential(
            nn.LayerNorm(model_dim),
            nn.Linear(model_dim, model_dim // 2),
            nn.GELU(),
            nn.Linear(model_dim // 2, 1),
        )
        self.out_dim = model_dim

    def forward(self, frame_features: torch.Tensor):
        """
        frame_features: [B,T,C]
        returns:
            context: [B,D]
            alpha: [B,T], sum=1, for weighted aggregation and Grad-CAM fusion
            beta: [B,T], sigmoid activation, for coverage and important frame detection
            seq: [B,T,D]
        """
        x = self.proj(frame_features)
        x = self.pos(x)
        seq = self.encoder(x)
        logits = self.att_head(seq).squeeze(-1)
        beta = torch.sigmoid(logits)
        alpha = beta / (beta.sum(dim=1, keepdim=True) + 1e-8)
        context = torch.sum(seq * alpha.unsqueeze(-1), dim=1)
        return context, alpha, beta, seq


class FinalAttentionGatedGradCAMModel(nn.Module):
    """
    Explainable-by-design video classifier.
    Backbone predicts action after gated spatio-temporal filtering.
    """
    def __init__(self, cfg: CFG, num_classes: int = 101):
        super().__init__()
        self.cfg = cfg
        self.backbone_name = cfg.backbone_name
        self.num_classes = num_classes

        if cfg.backbone_name == "s3d":
            weights = S3D_Weights.DEFAULT if cfg.pretrained else None
            base = s3d(weights=weights)
            self.features = base.features
            feat_dim = 1024
        elif cfg.backbone_name == "r3d_18":
            weights = R3D_18_Weights.DEFAULT if cfg.pretrained else None
            base = r3d_18(weights=weights)
            self.features = nn.Sequential(base.stem, base.layer1, base.layer2, base.layer3, base.layer4)
            feat_dim = 512
        else:
            raise ValueError("cfg.backbone_name must be 's3d' or 'r3d_18'")

        self.feat_dim = feat_dim
        self.spatial_gate = SpatialGate3D(feat_dim)
        if cfg.temporal_module == "lstm_attention":
            self.temporal_gate = TemporalLSTMAttentionGate(
                in_channels=feat_dim,
                hidden_dim=cfg.lstm_hidden_dim,
                bidirectional=cfg.lstm_bidirectional,
                dropout=cfg.transformer_dropout,
            )
        elif cfg.temporal_module == "transformer":
            self.temporal_gate = TemporalTransformerGate(
                in_channels=feat_dim,
                model_dim=cfg.temporal_hidden_dim,
                n_heads=cfg.transformer_heads,
                n_layers=cfg.transformer_layers,
                dropout=cfg.transformer_dropout,
            )
        else:
            raise ValueError("temporal_module must be 'lstm_attention' or 'transformer'")

        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(self.temporal_gate.out_dim, num_classes),
        )

    def forward(self, x: torch.Tensor, return_explanations: bool = True):
        feats = self.features(x)  # [B,C,T',H',W']
        spatial = self.spatial_gate(feats)
        gated = feats * spatial

        frame_features = gated.mean(dim=(3, 4)).permute(0, 2, 1)  # [B,T',C]
        context, alpha, beta, seq = self.temporal_gate(frame_features)
        logits = self.classifier(context)

        if not return_explanations:
            return logits

        return {
            "logits": logits,
            "features": feats,
            "gated_features": gated,
            "spatial_gate": spatial,
            "temporal_alpha": alpha,
            "temporal_beta": beta,
            "temporal_seq": seq,
        }


## 5. Contraintes XAI et fonction de perte mixte

Le modèle est entraîné avec une perte mixte combinant :

- perte de classification ;
- parcimonie spatiale ;
- régularisation spatiale ;
- régularisation temporelle ;
- contrainte de couverture temporelle ;
- alignement dynamique via optical flow ;
- fidélité par suppression/insertion ;
- contrainte causale légère.
 
L'objectif n'est pas uniquement d'obtenir une bonne accuracy, mais de produire une explication plus fidèle, stable et cohérente avec la dynamique vidéo.
 
Un mécanisme de **warm-up XAI** est utilisé : au début de l'entraînement, le modèle apprend principalement à classifier, puis les contraintes d'explicabilité sont activées progressivement.


In [6]:

# Explanation constraints and mixed loss

def total_variation_3d(mask: torch.Tensor) -> torch.Tensor:
    tv_t = torch.abs(mask[:, :, 1:] - mask[:, :, :-1]).mean() if mask.shape[2] > 1 else torch.tensor(0.0, device=mask.device)
    tv_h = torch.abs(mask[:, :, :, 1:] - mask[:, :, :, :-1]).mean()
    tv_w = torch.abs(mask[:, :, :, :, 1:] - mask[:, :, :, :, :-1]).mean()
    return tv_t + tv_h + tv_w


def temporal_entropy(alpha: torch.Tensor) -> torch.Tensor:
    ent = -(alpha * torch.log(alpha + 1e-8)).sum(dim=1)
    ent = ent / math.log(alpha.shape[1] + 1e-8)
    return ent.mean()


def temporal_smoothness(beta: torch.Tensor) -> torch.Tensor:
    if beta.shape[1] <= 1:
        return torch.tensor(0.0, device=beta.device)
    return torch.abs(beta[:, 1:] - beta[:, :-1]).mean()


def temporal_coverage_loss(beta: torch.Tensor, target_ratio: float) -> torch.Tensor:
    """
    Prevents too small attention segments.
    beta is sigmoid attention [0,1], not normalized.
    """
    coverage = beta.mean(dim=1)
    return ((coverage - target_ratio) ** 2).mean()


def motion_alignment_loss(alpha: torch.Tensor, temporal_flow: torch.Tensor) -> torch.Tensor:
    """
    Aligns temporal attention with optical-flow motion profile.
    """
    flow = F.interpolate(temporal_flow.unsqueeze(1), size=alpha.shape[1], mode="linear", align_corners=False).squeeze(1)
    flow = flow / (flow.sum(dim=1, keepdim=True) + 1e-8)
    return F.mse_loss(alpha, flow)


def soft_iou_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    pred, target: [B,1,T,H,W] in [0,1].
    """
    inter = (pred * target).sum(dim=(1, 2, 3, 4))
    union = pred.sum(dim=(1, 2, 3, 4)) + target.sum(dim=(1, 2, 3, 4)) - inter
    return (1.0 - (inter + 1e-6) / (union + 1e-6)).mean()


def build_spatiotemporal_mask(
    spatial_gate: torch.Tensor,
    beta: torch.Tensor,
    input_size: Tuple[int, int, int],
    topk_temporal_ratio: float,
    topk_spatial_ratio: float,
) -> torch.Tensor:
    """
    Builds aggressive input-level mask from learned spatial and temporal gates.
    spatial_gate: [B,1,Tf,Hf,Wf]
    beta: [B,Tf]
    input_size: (T,H,W)
    returns mask [B,1,T,H,W]
    """
    B, _, Tf, Hf, Wf = spatial_gate.shape
    T, H, W = input_size

    kt = max(1, int(Tf * topk_temporal_ratio))
    temp_idx = torch.topk(beta, k=kt, dim=1).indices
    temp_mask = torch.zeros_like(beta)
    temp_mask.scatter_(1, temp_idx, 1.0)
    temp_mask = temp_mask.view(B, 1, Tf, 1, 1)

    flat = spatial_gate.view(B, 1, Tf, Hf * Wf)
    ks = max(1, int(Hf * Wf * topk_spatial_ratio))
    sp_idx = torch.topk(flat, k=ks, dim=-1).indices
    sp_mask = torch.zeros_like(flat)
    sp_mask.scatter_(-1, sp_idx, 1.0)
    sp_mask = sp_mask.view(B, 1, Tf, Hf, Wf)

    mask_feat = temp_mask * sp_mask
    mask = F.interpolate(mask_feat, size=(T, H, W), mode="trilinear", align_corners=False)
    mask = (mask > 0.25).float()
    return mask


def input_level_fidelity_loss(
    model: nn.Module,
    video: torch.Tensor,
    logits: torch.Tensor,
    spatial_gate: torch.Tensor,
    beta: torch.Tensor,
    labels: torch.Tensor,
    cfg: CFG,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    Stronger fidelity:
        insertion: important region/frames alone should preserve clean distribution.
        deletion: removing important region/frames should suppress predicted/true class.
        strong causal: target confidence must drop by margin after deletion.
    """
    B, C, T, H, W = video.shape
    with torch.no_grad():
        clean_probs = torch.softmax(logits, dim=1)
        pred_idx = clean_probs.argmax(dim=1)
        target_idx = labels

    mask = build_spatiotemporal_mask(
        spatial_gate=spatial_gate.detach(),
        beta=beta.detach(),
        input_size=(T, H, W),
        topk_temporal_ratio=cfg.topk_temporal_ratio,
        topk_spatial_ratio=cfg.topk_spatial_ratio,
    )

    if cfg.deletion_fill == "mean":
        fill = video.mean(dim=(2, 3, 4), keepdim=True)
    else:
        fill = torch.zeros_like(video)

    deleted = video * (1.0 - mask) + fill * mask
    inserted = video * mask + fill * (1.0 - mask)

    logits_deleted = model(deleted, return_explanations=False)
    logits_inserted = model(inserted, return_explanations=False)

    # Sufficiency / insertion: selected evidence alone should reconstruct the clean decision distribution.
    insertion_kl = F.kl_div(
        F.log_softmax(logits_inserted, dim=1),
        clean_probs.detach(),
        reduction="batchmean",
    )

    # Deletion: selected evidence removed should not support the predicted class.
    deletion_target = clean_probs.detach().clone()
    deletion_target.scatter_(1, pred_idx.view(-1, 1), 0.0)
    deletion_bce = F.binary_cross_entropy_with_logits(logits_deleted, deletion_target)

    # Strong causal drop: confidence of the true class should drop after deletion.
    clean_target_prob = clean_probs.gather(1, target_idx.view(-1, 1)).squeeze(1).detach()
    deleted_target_prob = torch.softmax(logits_deleted, dim=1).gather(1, target_idx.view(-1, 1)).squeeze(1)
    strong_causal = F.relu(deleted_target_prob - clean_target_prob + cfg.fidelity_margin).mean()

    loss = cfg.lambda_sufficiency * insertion_kl + deletion_bce + cfg.lambda_strong_causal * strong_causal

    parts = {
        "input_insertion_kl": float(insertion_kl.detach().cpu()),
        "input_deletion_bce": float(deletion_bce.detach().cpu()),
        "strong_causal": float(strong_causal.detach().cpu()),
    }
    return loss, parts


def get_xai_scale(epoch: int, cfg: CFG) -> float:
    """Progressively activates XAI constraints after classification warm-up."""
    if epoch < cfg.xai_warmup_epochs:
        return 0.0
    ramp_pos = epoch - cfg.xai_warmup_epochs + 1
    return float(min(1.0, ramp_pos / max(1, cfg.xai_ramp_epochs)))


def mixed_loss(outputs: Dict, labels: torch.Tensor, video: torch.Tensor, model: nn.Module, cfg: CFG, epoch: int = 0):
    logits = outputs["logits"]
    spatial = outputs["spatial_gate"]
    alpha = outputs["temporal_alpha"]
    beta = outputs["temporal_beta"]

    pred_loss = F.cross_entropy(logits, labels)

    if cfg.use_optical_flow:
        temporal_flow, spatial_flow, binary_focus = compute_optical_flow_targets(
            video.detach(),
            out_hw=(spatial.shape[-2], spatial.shape[-1]),
            flow_size=cfg.flow_size,
            topk_ratio=cfg.flow_topk_ratio,
        )
    else:
        temporal_flow, spatial_flow, binary_focus = compute_frame_diff_targets(
            video.detach(),
            out_hw=(spatial.shape[-2], spatial.shape[-1]),
            topk_ratio=cfg.flow_topk_ratio,
        )

    # Match feature time dimension if needed.
    if binary_focus.shape[2] != spatial.shape[2]:
        binary_focus = F.interpolate(binary_focus, size=spatial.shape[2:], mode="trilinear", align_corners=False)
    if spatial_flow.shape[2] != spatial.shape[2]:
        spatial_flow = F.interpolate(spatial_flow, size=spatial.shape[2:], mode="trilinear", align_corners=False)

    spatial_sparse = spatial.mean()
    spatial_tv = total_variation_3d(spatial)
    temp_ent = temporal_entropy(alpha)
    temp_smooth = temporal_smoothness(beta)
    temp_cov = temporal_coverage_loss(beta, cfg.target_temporal_coverage)
    motion_align = motion_alignment_loss(alpha, temporal_flow)
    spatial_focus = soft_iou_loss(spatial, binary_focus)

    fidelity, fid_parts = input_level_fidelity_loss(
        model=model,
        video=video,
        logits=logits,
        spatial_gate=spatial,
        beta=beta,
        labels=labels,
        cfg=cfg,
    )

    xai_scale = get_xai_scale(epoch, cfg)
    xai_loss = (
        cfg.lambda_spatial_sparse * spatial_sparse
        + cfg.lambda_spatial_tv * spatial_tv
        + cfg.lambda_temporal_entropy * temp_ent
        + cfg.lambda_temporal_smooth * temp_smooth
        + cfg.lambda_temporal_coverage * temp_cov
        + cfg.lambda_motion_align * motion_align
        + cfg.lambda_spatial_focus * spatial_focus
        + cfg.lambda_input_fidelity * fidelity
    )
    total = pred_loss + xai_scale * xai_loss

    logs = {
        "loss_total": float(total.detach().cpu()),
        "loss_pred": float(pred_loss.detach().cpu()),
        "xai_scale": float(xai_scale),
        "spatial_sparse": float(spatial_sparse.detach().cpu()),
        "spatial_tv": float(spatial_tv.detach().cpu()),
        "temporal_entropy": float(temp_ent.detach().cpu()),
        "temporal_smooth": float(temp_smooth.detach().cpu()),
        "temporal_coverage": float(temp_cov.detach().cpu()),
        "motion_align_optical_flow": float(motion_align.detach().cpu()),
        "spatial_focus_iou_loss": float(spatial_focus.detach().cpu()),
        "input_fidelity": float(fidelity.detach().cpu()),
        **fid_parts,
    }
    return total, logs

## 6. Génération des explications Grad-CAM spatio-temporelles

Cette section définit la classe `VideoGradCAM` et les fonctions de visualisation.

L'explication finale combine :

$$\text{Explication finale} = \text{Grad-CAM} \times \text{Spatial Gate} \times \text{Temporal Attention}$$

La vidéo expliquée contient :
 
 - une heatmap superposée à la vidéo originale ;
 - un encadrement rouge sur les frames importantes ;
 - la classe prédite et la confiance du modèle ;
 - une description textuelle de la dynamique détectée.


In [7]:
# Grad-CAM fused with learned gates

class VideoGradCAM:
    def __init__(self, model: nn.Module):
        self.model = model

    def __call__(self, video: torch.Tensor, class_idx: Optional[int] = None):
        """
        video: [1,C,T,H,W]

        Correction importante :
        - Le modèle reste en eval() pendant Grad-CAM.
        - cuDNN est désactivé uniquement pendant le forward/backward Grad-CAM
          afin d'éviter l'erreur :
          "cudnn RNN backward can only be called in training mode".
        - Cela évite d'activer Dropout/BatchNorm en mode train pendant l'évaluation,
          ce qui faussait l'accuracy dans evaluate_full_split.

        returns:
            heatmap [T,H,W]
            alpha [T_feature]
            beta [T_feature]
            pred_idx
            prob
        """
        device = next(self.model.parameters()).device
        was_training = self.model.training
        self.model.eval()

        video = video.clone().detach().to(device)
        video.requires_grad_(True)

        # cuDNN désactivé seulement pour permettre le backward dans le LSTM en mode eval.
        with torch.backends.cudnn.flags(enabled=False):
            out = self.model(video, return_explanations=True)
            logits = out["logits"]
            probs = torch.softmax(logits, dim=1)

            if class_idx is None:
                class_idx = int(probs.argmax(dim=1).item())

            score = logits[:, class_idx].sum()
            feats = out["gated_features"]

            self.model.zero_grad(set_to_none=True)
            grads = torch.autograd.grad(score, feats, retain_graph=False, create_graph=False)[0]

            weights = grads.mean(dim=(3, 4), keepdim=True)
            cam = F.relu((weights * feats).sum(dim=1, keepdim=True))

            spatial = out["spatial_gate"].detach()
            alpha = out["temporal_alpha"].detach()
            beta = out["temporal_beta"].detach()

            alpha_5d = alpha.view(1, 1, alpha.shape[1], 1, 1)
            if alpha.shape[1] != cam.shape[2]:
                alpha_5d = F.interpolate(
                    alpha.view(1, 1, -1),
                    size=cam.shape[2],
                    mode="linear",
                    align_corners=False
                ).view(1, 1, cam.shape[2], 1, 1)

            fused = cam * spatial * alpha_5d
            fused = fused - fused.min()
            fused = fused / (fused.max() + 1e-8)

            fused_up = F.interpolate(
                fused,
                size=(video.shape[2], video.shape[3], video.shape[4]),
                mode="trilinear",
                align_corners=False,
            )

            heatmap = fused_up[0, 0].detach().cpu().numpy()
            prob = float(probs[0, class_idx].detach().cpu())

        if was_training:
            self.model.train()
        else:
            self.model.eval()

        return heatmap, alpha[0].detach().cpu(), beta[0].detach().cpu(), int(class_idx), prob


In [23]:
# Visualization

def overlay_heatmap(frame_rgb: np.ndarray, heat: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heat_uint = np.uint8(255 * np.clip(heat, 0, 1))
    color = cv2.applyColorMap(heat_uint, cv2.COLORMAP_JET)
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(frame_rgb, 1 - alpha, color, alpha, 0)


def save_explanation_video(
    video_tensor: torch.Tensor,
    heatmap: np.ndarray,
    temporal_beta: torch.Tensor,
    pred_text: str,
    prob: float,
    dynamics_text: str,
    save_path: str,
    fps: int = 8,
):
    frames = denormalize_video(video_tensor)
    T, H, W, _ = frames.shape

    beta = F.interpolate(temporal_beta.view(1, 1, -1), size=T, mode="linear", align_corners=False).view(-1).numpy()
    threshold = np.percentile(beta, 75)
    important = beta >= threshold

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(save_path, fourcc, fps, (W, H))

    for t in range(T):
        frame = overlay_heatmap(frames[t], heatmap[t])
        if important[t]:
            cv2.rectangle(frame, (3, 3), (W - 4, H - 4), (255, 0, 0), 5)

        txt1 = f"{pred_text} ({prob:.2f})"
        cv2.putText(frame, txt1, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
        #cv2.putText(frame, txt2, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    writer.release()


def save_original_clip(sample, save_path: str, fps: int = 8):
    frames = denormalize_video(sample["video"])
    T, H, W, _ = frames.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(save_path, fourcc, fps, (W, H))
    for fr in frames:
        writer.write(cv2.cvtColor(fr, cv2.COLOR_RGB2BGR))
    writer.release()
    return save_path


def convert_to_web_mp4(input_path: str, output_path: str):
    os.system(f'ffmpeg -y -i "{input_path}" -vcodec libx264 -pix_fmt yuv420p "{output_path}" -loglevel quiet')
    return output_path


def show_original_and_explanation(report, sample=None, width: int = 750):
    os.makedirs("./xai_outputs_v4/web", exist_ok=True)

    if sample is not None:
        original_path = "./xai_outputs_v4/web/original_tmp.mp4"
        save_original_clip(sample, original_path)
        original_web = "./xai_outputs_v4/web/original_web.mp4"
        convert_to_web_mp4(original_path, original_web)
        print("Vidéo originale")
        display(Video(original_web, embed=True, width=width))

    explained_web = "./xai_outputs_v4/web/explained_web.mp4"
    convert_to_web_mp4(report["output_video"], explained_web)
    print("Vidéo expliquée — Grad-CAM spatio-temporel")
    display(Video(explained_web, embed=True, width=width))

    if report.get("dynamic_output_video") is not None:
        dynamic_web = "./xai_outputs_v4/web/dynamic_web.mp4"
        convert_to_web_mp4(report["dynamic_output_video"], dynamic_web)
        print("Vidéo expliquée — Axe dynamique par Optical Flow")
        display(Video(dynamic_web, embed=True, width=width))

    display(HTML(f"""
    <div style="font-family: Arial; max-width: 900px;">
      <h3>Explication complète</h3>
      <ul>
        <li><b>Ground truth:</b> {report.get("ground_truth")}</li>
        <li><b>Prediction:</b> {report.get("prediction")}</li>
        <li><b>Correct:</b> {report.get("correct")}</li>
        <li><b>Confidence:</b> {report.get("probability"):.4f}</li>
        <li><b>Dynamique:</b> {report.get("important_dynamics")}</li>
      </ul>
      <h4>Métriques</h4>
      <ul>
        <li><b>Deletion AUC:</b> {report.get("temporal_deletion_auc"):.4f} ; plus bas = meilleure causalité.</li>
        <li><b>Insertion AUC:</b> {report.get("temporal_insertion_auc"):.4f} ; plus haut = meilleure suffisance.</li>
        <li><b>Stabilité:</b> {report.get("temporal_stability"):.6f} ; plus bas = moins de flickering.</li>
        <li><b>Motion proxy:</b> {report.get("motion_localization_proxy"):.4f} ; plus haut = meilleur alignement dynamique.</li>
      </ul>
      <h4>Diagnostic dynamique visuel</h4>
      <ul>
        <li><b>Direction dominante:</b> {report.get("dynamic_dominant_motion_direction")}</li>
        <li><b>Corrélation Grad-CAM / mouvement:</b> {report.get("dynamic_motion_gradcam_correlation")}</li>
        <li><b>Mouvement utile dans les zones Grad-CAM:</b> {report.get("dynamic_useful_motion_ratio_percent")}%</li>
        <li><b>Alignement temporel mouvement-explication:</b> {report.get("dynamic_temporal_motion_alignment")}</li>
      </ul>
      <p>La première vidéo affiche Grad-CAM. La deuxième vidéo affiche l'axe dynamique : intensité du mouvement, flèches de direction et mouvement utile croisé avec Grad-CAM. Le cadre rouge indique les frames importantes.</p>
    </div>
    """))


### Visualisation de l'axe dynamique par Optical Flow

Cette section ajoute une visualisation complémentaire de l'axe **Comment ?** sans modifier l'architecture et sans réentraîner le modèle.

L'idée est de produire une deuxième vidéo explicative où la superposition ne représente plus directement Grad-CAM, mais l'intensité du mouvement estimée par **Optical Flow**.  
La carte dynamique peut aussi être croisée avec Grad-CAM afin de visualiser le **mouvement utile**, c'est-à-dire le mouvement situé dans les régions réellement importantes pour la décision du modèle.

Cette visualisation permet de vérifier qualitativement si le modèle regarde :
- des zones réellement en mouvement ;
- des mouvements cohérents avec l'action ;
- ou au contraire des mouvements parasites comme le fond, la caméra ou des objets non pertinents.


In [20]:

# ============================================================
# Visualisation dynamique par Optical Flow
# ============================================================

def compute_dense_optical_flow_maps(
    video_tensor: torch.Tensor,
    flow_resize: Optional[Tuple[int, int]] = None,
) -> Dict[str, np.ndarray]:
    """
    Calcule un champ de flux optique dense entre les frames successives.

    Entrée:
        video_tensor:
            tenseur vidéo [C,T,H,W], normalisé ou brut.

    Sortie:
        dictionnaire contenant :
            - magnitude_maps: [T,H,W], intensité du mouvement normalisée ;
            - angle_maps: [T,H,W], direction angulaire du mouvement ;
            - flow_u: [T,H,W], composante horizontale ;
            - flow_v: [T,H,W], composante verticale ;
            - temporal_energy: [T], énergie moyenne du mouvement par frame.

    Remarque:
        Cette fonction est utilisée seulement pour l'affichage dynamique.
        Elle ne modifie ni l'architecture ni les poids du modèle.
    """
    frames = denormalize_video(video_tensor)
    T, H, W, _ = frames.shape

    if flow_resize is None:
        flow_w, flow_h = W, H
    else:
        flow_h, flow_w = flow_resize

    gray_seq = []
    for fr in frames:
        gray = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY)
        if (flow_h, flow_w) != (H, W):
            gray = cv2.resize(gray, (flow_w, flow_h))
        gray_seq.append(gray)

    mag_list = []
    angle_list = []
    u_list = []
    v_list = []

    if T <= 1:
        z = np.zeros((H, W), dtype=np.float32)
        return {
            "magnitude_maps": z[None],
            "angle_maps": z[None],
            "flow_u": z[None],
            "flow_v": z[None],
            "temporal_energy": np.zeros(1, dtype=np.float32),
        }

    for t in range(T - 1):
        flow = cv2.calcOpticalFlowFarneback(
            gray_seq[t],
            gray_seq[t + 1],
            None,
            pyr_scale=0.5,
            levels=3,
            winsize=15,
            iterations=3,
            poly_n=5,
            poly_sigma=1.2,
            flags=0,
        )

        u = flow[..., 0].astype(np.float32)
        v = flow[..., 1].astype(np.float32)
        mag, angle = cv2.cartToPolar(u, v, angleInDegrees=False)

        if (flow_h, flow_w) != (H, W):
            u = cv2.resize(u, (W, H))
            v = cv2.resize(v, (W, H))
            mag = cv2.resize(mag, (W, H))
            angle = cv2.resize(angle, (W, H))

        mag_list.append(mag.astype(np.float32))
        angle_list.append(angle.astype(np.float32))
        u_list.append(u.astype(np.float32))
        v_list.append(v.astype(np.float32))

    # On duplique la première carte pour avoir T cartes, une par frame.
    mag_list = [mag_list[0]] + mag_list
    angle_list = [angle_list[0]] + angle_list
    u_list = [u_list[0]] + u_list
    v_list = [v_list[0]] + v_list

    magnitude_maps = np.stack(mag_list).astype(np.float32)
    angle_maps = np.stack(angle_list).astype(np.float32)
    flow_u = np.stack(u_list).astype(np.float32)
    flow_v = np.stack(v_list).astype(np.float32)

    temporal_energy = magnitude_maps.mean(axis=(1, 2))
    magnitude_maps_norm = magnitude_maps / (magnitude_maps.max() + 1e-8)

    return {
        "magnitude_maps": magnitude_maps_norm,
        "angle_maps": angle_maps,
        "flow_u": flow_u,
        "flow_v": flow_v,
        "temporal_energy": temporal_energy.astype(np.float32),
    }


def resize_heatmap_to_frames(heatmap: np.ndarray, T: int, H: int, W: int) -> np.ndarray:
    """
    Aligne une heatmap spatio-temporelle sur la taille de la vidéo originale.
    """
    hm = np.asarray(heatmap, dtype=np.float32)

    if hm.ndim == 2:
        hm = np.repeat(hm[None], T, axis=0)

    # Alignement temporel si nécessaire.
    if hm.shape[0] != T:
        hm_t = torch.tensor(hm).unsqueeze(0).unsqueeze(0).float()
        hm_t = F.interpolate(
            hm_t,
            size=(T, hm.shape[1], hm.shape[2]),
            mode="trilinear",
            align_corners=False,
        )
        hm = hm_t.squeeze().numpy()

    out = []
    for t in range(T):
        h = hm[t]
        h = h - h.min()
        h = h / (h.max() + 1e-8)
        out.append(cv2.resize(h, (W, H)))

    return np.stack(out).astype(np.float32)


def compute_dynamic_alignment_diagnostic(
    heatmap: np.ndarray,
    magnitude_maps: np.ndarray,
    temporal_scores: torch.Tensor,
) -> Dict[str, Any]:
    """
    Calcule des diagnostics simples entre :
        - l'importance Grad-CAM ;
        - l'intensité du mouvement Optical Flow ;
        - l'importance temporelle finale.

    Ces métriques ne prouvent pas seules la causalité.
    Elles indiquent si les zones expliquées sont cohérentes avec le mouvement observé.
    """
    T, H, W = magnitude_maps.shape
    hm = resize_heatmap_to_frames(heatmap, T, H, W)

    # Corrélation pixel-wise Grad-CAM / mouvement.
    h_flat = hm.reshape(-1)
    m_flat = magnitude_maps.reshape(-1)

    if np.std(h_flat) < 1e-8 or np.std(m_flat) < 1e-8:
        corr = 0.0
    else:
        corr = float(np.corrcoef(h_flat, m_flat)[0, 1])

    # Ratio du mouvement situé dans les zones Grad-CAM fortes.
    active = hm >= np.percentile(hm, 75)
    total_motion = float(magnitude_maps.sum() + 1e-8)
    useful_motion = float((magnitude_maps * active).sum())
    useful_motion_ratio = useful_motion / total_motion

    # Alignement temporel : frames importantes vs énergie mouvement.
    s = temporal_scores.detach().cpu().float()
    if s.numel() != T:
        s = F.interpolate(s.view(1, 1, -1), size=T, mode="linear", align_corners=False).view(-1)
    s_np = s.numpy()
    s_np = s_np / (s_np.sum() + 1e-8)

    e = magnitude_maps.mean(axis=(1, 2))
    e_norm = e / (e.max() + 1e-8)

    temporal_alignment = float((s_np * e_norm).sum())

    return {
        "motion_gradcam_correlation": round(corr, 4),
        "useful_motion_ratio_percent": round(useful_motion_ratio * 100, 2),
        "temporal_motion_alignment": round(temporal_alignment, 4),
    }


def dominant_motion_direction(
    flow_u: np.ndarray,
    flow_v: np.ndarray,
    magnitude_maps: np.ndarray,
    heatmap: Optional[np.ndarray] = None,
) -> str:
    """
    Estime la direction dominante du mouvement dans les zones les plus importantes.
    """
    T, H, W = magnitude_maps.shape

    if heatmap is not None:
        hm = resize_heatmap_to_frames(heatmap, T, H, W)
        mask = (hm >= np.percentile(hm, 75)) & (magnitude_maps >= np.percentile(magnitude_maps, 75))
    else:
        mask = magnitude_maps >= np.percentile(magnitude_maps, 75)

    if mask.sum() < 10:
        return "indéterminée"

    u_mean = float(flow_u[mask].mean())
    v_mean = float(flow_v[mask].mean())

    if abs(u_mean) < 1e-3 and abs(v_mean) < 1e-3:
        return "mouvement quasi stable"

    if abs(u_mean) >= abs(v_mean):
        return "droite" if u_mean > 0 else "gauche"
    else:
        return "bas" if v_mean > 0 else "haut"


def save_dynamic_explanation_video(
    video_tensor: torch.Tensor,
    heatmap: np.ndarray,
    temporal_scores: torch.Tensor,
    pred_text: str,
    prob: float,
    dynamics: Dict[str, Any],
    save_path: str,
    fps: int = 8,
    mode: str = "motion_guided",
):
    """
    Sauvegarde une vidéo d'explication dynamique.

    Modes:
        - "motion":
            affiche uniquement l'intensité du mouvement Optical Flow.
        - "motion_guided":
            affiche le mouvement pondéré par Grad-CAM.
            Ce mode montre le mouvement utile dans les régions importantes.
        - "motion_plus_gradcam":
            mélange l'intensité du mouvement et Grad-CAM.

    Cette vidéo complète la vidéo Grad-CAM classique :
        - Grad-CAM explique surtout où ;
        - cette vidéo montre comment le mouvement se distribue.
    """
    frames = denormalize_video(video_tensor)
    T, H, W, _ = frames.shape

    flow_data = compute_dense_optical_flow_maps(video_tensor)
    mag = flow_data["magnitude_maps"]
    u = flow_data["flow_u"]
    v = flow_data["flow_v"]

    hm = resize_heatmap_to_frames(heatmap, T, H, W)

    if mode == "motion":
        dynamic_maps = mag
    elif mode == "motion_plus_gradcam":
        dynamic_maps = 0.60 * mag + 0.40 * hm
    else:
        # Mouvement utile : mouvement présent dans les zones importantes Grad-CAM.
        dynamic_maps = mag * (0.35 + 0.65 * hm)

    dynamic_maps = dynamic_maps - dynamic_maps.min()
    dynamic_maps = dynamic_maps / (dynamic_maps.max() + 1e-8)

    scores = temporal_scores.detach().cpu().float()
    if scores.numel() != T:
        scores = F.interpolate(scores.view(1, 1, -1), size=T, mode="linear", align_corners=False).view(-1)
    scores_np = scores.numpy()
    important_thr = np.percentile(scores_np, 75)

    direction_label = dominant_motion_direction(u, v, mag, heatmap=heatmap)
    diagnostic = compute_dynamic_alignment_diagnostic(heatmap, mag, temporal_scores)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(save_path, fourcc, fps, (W, H))

    grid_step = max(16, min(H, W) // 10)

    for t in range(T):
        frame = frames[t].copy()

        dyn_uint = np.uint8(255 * np.clip(dynamic_maps[t], 0, 1))
        color = cv2.applyColorMap(dyn_uint, cv2.COLORMAP_TURBO)
        color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
        frame = cv2.addWeighted(frame, 0.55, color, 0.45, 0)

        # Contour des zones dynamiques fortes.
        motion_mask = dynamic_maps[t] >= np.percentile(dynamic_maps[t], 85)
        contours, _ = cv2.findContours(
            motion_mask.astype(np.uint8),
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )
        cv2.drawContours(frame, contours, -1, (255, 255, 255), 1)

        # Flèches directionnelles échantillonnées.
        for yy in range(grid_step // 2, H, grid_step):
            for xx in range(grid_step // 2, W, grid_step):
                if mag[t, yy, xx] < np.percentile(mag[t], 85):
                    continue

                dx = int(np.clip(u[t, yy, xx] * 2.0, -18, 18))
                dy = int(np.clip(v[t, yy, xx] * 2.0, -18, 18))

                if abs(dx) + abs(dy) < 2:
                    continue

                cv2.arrowedLine(
                    frame,
                    (xx, yy),
                    (xx + dx, yy + dy),
                    (255, 255, 255),
                    1,
                    tipLength=0.35,
                )

        if scores_np[t] >= important_thr:
            cv2.rectangle(frame, (3, 3), (W - 4, H - 4), (255, 0, 0), 5)

        txt1 = f"{pred_text} ({prob:.2f})"
        txt2 = f"Importance: {scores_np[t]:.3f}"

        cv2.putText(frame, txt1, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, txt2, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 2, cv2.LINE_AA)
        
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    writer.release()

    return {
        "dynamic_video_path": save_path,
        "dominant_motion_direction": direction_label,
        **diagnostic,
    }


def show_dynamic_explanation(report: Dict[str, Any], width: int = 750):
    """
    Affiche uniquement la vidéo dynamique si elle est disponible.
    """
    dynamic_path = report.get("dynamic_output_video")
    if dynamic_path is None:
        print("Aucune vidéo dynamique n'est disponible dans le rapport.")
        return

    os.makedirs("./xai_outputs_v4/web", exist_ok=True)
    dynamic_web = "./xai_outputs_v4/web/dynamic_web.mp4"
    convert_to_web_mp4(dynamic_path, dynamic_web)

    print("Vidéo d'explication dynamique")
    display(Video(dynamic_web, embed=True, width=width))

    display(HTML(f"""
    <div style="font-family: Arial; max-width: 900px;">
      <h3>Diagnostic dynamique</h3>
      <ul>
        <li><b>Direction dominante:</b> {report.get("dynamic_dominant_motion_direction")}</li>
        <li><b>Corrélation Grad-CAM / mouvement:</b> {report.get("dynamic_motion_gradcam_correlation")}</li>
        <li><b>Mouvement utile dans les zones Grad-CAM:</b> {report.get("dynamic_useful_motion_ratio_percent")}%</li>
        <li><b>Alignement temporel mouvement-explication:</b> {report.get("dynamic_temporal_motion_alignment")}</li>
      </ul>
      <p>
      Cette vidéo ne remplace pas Grad-CAM. Elle montre si les régions expliquées correspondent
      réellement à des zones en mouvement, ce qui aide à détecter les biais de contexte ou les mouvements parasites.
      </p>
    </div>
    """))


## 7. Métriques d'évaluation

Les métriques utilisées sont alignées avec l'objectif du PFE : prédiction + explication fidèle.

Les métriques principales sont :

 - **Accuracy** : performance de classification ;
 - **Temporal Deletion AUC** : mesure si les frames importantes sont nécessaires à la décision ;
 - **Temporal Insertion AUC** : mesure si les frames importantes sont suffisantes pour reconstruire la décision ;
 - **Temporal Stability** : mesure la stabilité des heatmaps entre frames successives ;
 - **Motion Localization Proxy** : mesure l'alignement entre les régions importantes et le mouvement vidéo.


In [10]:
# Evaluation metrics

def accuracy(logits, labels):
    return (logits.argmax(dim=1) == labels).float().mean().item()


def heatmap_temporal_stability(heatmap: np.ndarray) -> float:
    """
    Évalue la stabilité temporelle de la heatmap entre frames successives.
    Plus la valeur est faible, moins l'explication clignote dans le temps.
    """
    if heatmap.shape[0] <= 1:
        return 0.0
    return float(np.abs(heatmap[1:] - heatmap[:-1]).mean())


def heatmap_temporal_importance(heatmap: np.ndarray, eps: float = 1e-8) -> torch.Tensor:
    """
    Convertit la heatmap spatio-temporelle finale en score temporel par frame.

    Pourquoi cette métrique ?
    ------------------------
    Dans cette architecture, la porte temporelle beta peut devenir presque uniforme
    à cause de la contrainte de couverture. Si on utilise seulement beta pour
    deletion/insertion, l'ordre temporel devient peu informatif.

    La heatmap finale contient déjà :
        Grad-CAM x porte spatiale x attention temporelle.

    Elle représente donc mieux l'importance finale réellement affichée dans la vidéo.
    """
    hm = np.asarray(heatmap, dtype=np.float32)
    if hm.ndim != 3:
        raise ValueError("heatmap doit être de forme [T,H,W]")

    scores = hm.mean(axis=(1, 2))
    scores = scores - scores.min()
    scores = scores / (scores.sum() + eps)

    if not np.isfinite(scores).all() or scores.sum() <= eps:
        scores = np.ones((hm.shape[0],), dtype=np.float32) / max(1, hm.shape[0])

    return torch.tensor(scores, dtype=torch.float32)


def localization_motion_proxy(heatmap: np.ndarray, video: torch.Tensor, cfg: CFG) -> float:
    """
    Mesure l'alignement entre la heatmap explicative et les zones de mouvement.
    Plus la valeur est élevée, plus l'explication regarde des régions dynamiques.
    """
    with torch.no_grad():
        temporal_flow, spatial_flow, _ = compute_optical_flow_targets(
            video.unsqueeze(0).to(cfg.device),
            out_hw=(heatmap.shape[1], heatmap.shape[2]),
            flow_size=cfg.flow_size,
            topk_ratio=cfg.flow_topk_ratio,
        )
    flow = spatial_flow[0, 0].detach().cpu().numpy()

    if flow.shape[0] != heatmap.shape[0]:
        flow_t = torch.tensor(flow).unsqueeze(0).unsqueeze(0)
        flow = F.interpolate(flow_t, size=heatmap.shape, mode="trilinear", align_corners=False)[0, 0].numpy()

    hm = heatmap / (heatmap.max() + 1e-8)
    fl = flow / (flow.max() + 1e-8)
    return float((hm * fl).sum() / (hm.sum() + 1e-8))


@torch.no_grad()
def temporal_deletion_insertion_score(
    model: nn.Module,
    video: torch.Tensor,
    target_class: int,
    temporal_scores: torch.Tensor,
    steps: int = 8,
    fill_strategy: str = "mean",
):
    """
    Évalue la fidélité temporelle de l'explication.

    Correction importante :
    -----------------------
    L'ancienne version utilisait beta comme ordre temporel. Or beta peut être
    presque uniforme. Ici, on utilise temporal_scores issus de la heatmap finale.

    Deletion :
        on supprime progressivement les frames les plus importantes.
        Plus l'AUC est basse, plus l'explication est nécessaire.

    Insertion :
        on insère progressivement les frames les plus importantes.
        Plus l'AUC est haute, plus l'explication est suffisante.
    """
    model.eval()
    B, C, T, H, W = video.shape
    assert B == 1

    scores = temporal_scores.detach().float().to(video.device).view(-1)
    if scores.numel() != T:
        scores = F.interpolate(
            scores.view(1, 1, -1),
            size=T,
            mode="linear",
            align_corners=False
        ).view(-1)

    order = torch.argsort(scores, descending=True)

    if fill_strategy == "zero":
        fill = torch.zeros_like(video)
    elif fill_strategy == "mean":
        fill = video.mean(dim=(2, 3, 4), keepdim=True).expand_as(video)
    else:
        raise ValueError("fill_strategy doit être 'mean' ou 'zero'")

    deletion_probs, insertion_probs = [], []
    chunk = max(1, T // steps)

    for s in range(0, T + 1, chunk):
        idx = order[:s]
        vd = video.clone()
        vi = fill.clone()

        if s > 0:
            vd[:, :, idx] = fill[:, :, idx]
            vi[:, :, idx] = video[:, :, idx]

        pd_prob = torch.softmax(model(vd, return_explanations=False), dim=1)[0, target_class].item()
        pi_prob = torch.softmax(model(vi, return_explanations=False), dim=1)[0, target_class].item()

        deletion_probs.append(pd_prob)
        insertion_probs.append(pi_prob)

    deletion_auc = float(np.trapezoid(deletion_probs) / max(1, len(deletion_probs) - 1))
    insertion_auc = float(np.trapezoid(insertion_probs) / max(1, len(insertion_probs) - 1))
    return deletion_auc, insertion_auc, deletion_probs, insertion_probs


def _make_deletion_fill(video: torch.Tensor, strategy: str = "blur") -> torch.Tensor:
    """
    Construit une référence de remplacement pour les régions supprimées.

    - mean : remplace par la moyenne RGB globale de la vidéo.
    - zero : remplace par zéro.
    - blur : remplace par une version floutée spatialement, plus naturelle que zéro
      et plus agressive que la moyenne globale pour casser les détails locaux.
    """
    if strategy == "zero":
        return torch.zeros_like(video)
    if strategy == "mean":
        return video.mean(dim=(2, 3, 4), keepdim=True).expand_as(video)
    if strategy == "blur":
        # Flou spatial frame par frame : conserve une structure globale mais supprime les détails discriminants.
        return F.avg_pool3d(
            video,
            kernel_size=(1, 21, 21),
            stride=1,
            padding=(0, 10, 10),
        )
    raise ValueError("strategy doit être 'blur', 'mean' ou 'zero'")


def _upsample_heatmap_to_video(
    heatmap: np.ndarray,
    target_size: Tuple[int, int, int],
    device: str,
) -> torch.Tensor:
    """
    Redimensionne une heatmap [T,H,W] vers la taille vidéo [T,H,W].
    Retourne un tenseur normalisé [T,H,W].
    """
    T, H, W = target_size
    hm = np.asarray(heatmap, dtype=np.float32)
    hm = np.clip(hm, 0, None)

    hm_t = torch.tensor(hm, dtype=torch.float32, device=device).view(1, 1, *hm.shape)
    hm_t = F.interpolate(hm_t, size=(T, H, W), mode="trilinear", align_corners=False)[0, 0]

    hm_t = hm_t - hm_t.min()
    hm_t = hm_t / (hm_t.max() + 1e-8)
    return hm_t


def _binary_mask_from_top_heatmap(
    heatmap_video: torch.Tensor,
    top_ratio: float,
    dilation: int = 11,
) -> torch.Tensor:
    """
    Crée un masque binaire [T,H,W] contenant les positions les plus importantes
    selon la heatmap finale. Une dilatation spatiale est appliquée pour supprimer
    non seulement le pixel chaud, mais aussi son voisinage visuel.
    """
    top_ratio = float(np.clip(top_ratio, 0.0, 1.0))
    if top_ratio <= 0:
        return torch.zeros_like(heatmap_video)

    flat = heatmap_video.flatten()
    k = max(1, int(flat.numel() * top_ratio))
    threshold = torch.topk(flat, k=k, largest=True).values.min()
    mask = (heatmap_video >= threshold).float()

    if dilation and dilation > 1:
        if dilation % 2 == 0:
            dilation += 1
        pad = dilation // 2
        mask_5d = mask.view(1, 1, *mask.shape)
        # Dilatation spatiale seulement, pour éviter d'effacer artificiellement tout le temps.
        mask_5d = F.max_pool3d(mask_5d, kernel_size=(1, dilation, dilation), stride=1, padding=(0, pad, pad))
        mask = (mask_5d[0, 0] > 0).float()

    return mask


@torch.no_grad()
def spatiotemporal_deletion_insertion_score(
    model: nn.Module,
    video: torch.Tensor,
    target_class: int,
    heatmap: np.ndarray,
    steps: int = 8,
    max_top_ratio: float = 0.35,
    fill_strategy: str = "blur",
    dilation: int = 11,
):
    """
    Évalue la fidélité spatio-temporelle de l'explication finale.

    Différence avec la métrique temporelle :
    ---------------------------------------
    La métrique temporelle supprime des frames entières. Ici, on supprime
    progressivement les régions spatio-temporelles les plus importantes selon
    la heatmap finale Grad-CAM guidée.

    Cette version est plus adaptée à l'axe "où + quand" :
        - Deletion : les régions importantes sont supprimées.
        - Insertion : les régions importantes sont réintroduites depuis une vidéo floutée.

    Objectif :
        obtenir une mesure plus causale sans réentraîner le modèle.
    """
    model.eval()
    B, C, T, H, W = video.shape
    assert B == 1

    hm_video = _upsample_heatmap_to_video(heatmap, target_size=(T, H, W), device=video.device)
    fill = _make_deletion_fill(video, strategy=fill_strategy)

    deletion_probs, insertion_probs = [], []
    ratios = np.linspace(0.0, max_top_ratio, steps + 1)

    for ratio in ratios:
        mask_t = _binary_mask_from_top_heatmap(hm_video, top_ratio=float(ratio), dilation=dilation)
        mask = mask_t.view(1, 1, T, H, W)

        vd = video * (1.0 - mask) + fill * mask
        vi = fill * (1.0 - mask) + video * mask

        pd_prob = torch.softmax(model(vd, return_explanations=False), dim=1)[0, target_class].item()
        pi_prob = torch.softmax(model(vi, return_explanations=False), dim=1)[0, target_class].item()

        deletion_probs.append(pd_prob)
        insertion_probs.append(pi_prob)

    deletion_auc = float(np.trapezoid(deletion_probs, x=ratios) / max_top_ratio)
    insertion_auc = float(np.trapezoid(insertion_probs, x=ratios) / max_top_ratio)

    # Drop final : facile à interpréter dans le rapport.
    deletion_drop = float(deletion_probs[0] - deletion_probs[-1])
    insertion_gain = float(insertion_probs[-1] - insertion_probs[0])

    return deletion_auc, insertion_auc, deletion_drop, insertion_gain, deletion_probs, insertion_probs


## 8. Entraînement, validation et gestion des checkpoints

Cette section contient les fonctions d'entraînement et de validation.

Le notebook sauvegarde automatiquement :

- `last_checkpoint.pt` : dernier état d'entraînement ;
- `best_attention_gated_final_model.pt` : meilleur modèle selon la validation ;
- `batch_safety_checkpoint.pt` : sauvegarde de sécurité pendant une epoch.

Ces checkpoints permettent de reprendre l'entraînement après une coupure Kaggle sans repartir de zéro.


In [11]:
# Training, checkpointing, and recovery

def save_checkpoint(path, model, optimizer, epoch, best_acc, train_ds, cfg, history=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "epoch": epoch,
        "best_acc": best_acc,
        "class_to_idx": train_ds.class_to_idx,
        "cfg": cfg.__dict__,
        "history": history or [],
    }, path)
    print("Checkpoint sauvegardé:", path)


def find_available_checkpoint(output_dir="./xai_outputs_attention_gated_final"):
    candidates = [
        os.path.join(output_dir, "best_attention_gated_final_model.pt"),
        os.path.join(output_dir, "last_checkpoint.pt"),
        os.path.join(output_dir, "batch_safety_checkpoint.pt"),
    ]
    for p in candidates:
        if os.path.exists(p):
            print("Checkpoint utilisé:", p)
            return p
    raise FileNotFoundError(f"Aucun checkpoint trouvé dans {output_dir}")


def train_one_epoch(model, loader, optimizer, cfg, epoch, best_acc, train_ds, history):
    model.train()
    logs, accs = [], []

    for step, batch in enumerate(tqdm(loader, desc="train")):
        video = batch["video"].to(cfg.device)
        labels = batch["label"].to(cfg.device)

        outputs = model(video, return_explanations=True)
        loss, parts = mixed_loss(outputs, labels, video, model, cfg, epoch=epoch)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        logs.append(parts)
        accs.append(accuracy(outputs["logits"].detach(), labels))

        if (step + 1) % 500 == 0:
            save_checkpoint(
                os.path.join(cfg.output_dir, "batch_safety_checkpoint.pt"),
                model, optimizer, epoch, best_acc, train_ds, cfg, history
            )

    mean_logs = {k: float(np.mean([d[k] for d in logs])) for k in logs[0].keys()}
    mean_logs["acc"] = float(np.mean(accs))
    return mean_logs


@torch.no_grad()
def validate(model, loader, cfg):
    model.eval()
    losses, accs = [], []
    for batch in tqdm(loader, desc="valid"):
        video = batch["video"].to(cfg.device)
        labels = batch["label"].to(cfg.device)
        logits = model(video, return_explanations=False)
        losses.append(float(F.cross_entropy(logits, labels).cpu()))
        accs.append(accuracy(logits, labels))
    return {"val_loss": float(np.mean(losses)), "val_acc": float(np.mean(accs))}


def train_main(resume_from: Optional[str] = None):
    cfg = CFG()
    seed_everything(cfg.seed)
    os.makedirs(cfg.output_dir, exist_ok=True)

    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = build_loaders(cfg)

    model = FinalAttentionGatedGradCAMModel(cfg, num_classes=cfg.num_classes).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    start_epoch = 0
    best_acc = -1.0
    history = []

    if resume_from is not None and os.path.exists(resume_from):
        ckpt = torch.load(resume_from, map_location=cfg.device)
        model.load_state_dict(ckpt["model"], strict=True)
        if ckpt.get("optimizer") is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = int(ckpt.get("epoch", -1)) + 1
        best_acc = float(ckpt.get("best_acc", -1.0))
        history = ckpt.get("history", [])
        print(f"Reprise depuis epoch index {start_epoch}. Best val_acc précédent: {best_acc:.4f}")

    for epoch in range(start_epoch, cfg.epochs):
        tr = train_one_epoch(model, train_loader, optimizer, cfg, epoch, best_acc, train_ds, history)
        va = validate(model, val_loader, cfg)

        row = {"epoch": epoch + 1, **tr, **va}
        history.append(row)
        print(f"Epoch {epoch+1}/{cfg.epochs} | train={tr} | valid={va}")

        save_checkpoint(os.path.join(cfg.output_dir, "last_checkpoint.pt"), model, optimizer, epoch, best_acc, train_ds, cfg, history)

        if va["val_acc"] > best_acc:
            best_acc = va["val_acc"]
            save_checkpoint(os.path.join(cfg.output_dir, "best_attention_gated_final_model.pt"), model, optimizer, epoch, best_acc, train_ds, cfg, history)

        pd.DataFrame(history).to_csv(os.path.join(cfg.output_dir, "training_history.csv"), index=False)

    return model, optimizer, cfg


## 9. Inférence, explication et évaluation complète
 
Cette section contient les fonctions permettant de :

- charger un modèle entraîné ;
- expliquer une vidéo de test ;
- évaluer tout le split de test ;
- résumer les résultats globaux et par classe ;
- exporter les vidéos, métriques et figures.


In [12]:
# Single-video explanation and full split evaluation

def load_model_from_checkpoint(checkpoint_path: str):
    cfg = CFG()
    ckpt = torch.load(checkpoint_path, map_location=cfg.device)

    if "cfg" in ckpt:
        for k, v in ckpt["cfg"].items():
            if hasattr(cfg, k):
                setattr(cfg, k, v)

    cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

    model = FinalAttentionGatedGradCAMModel(cfg, num_classes=cfg.num_classes).to(cfg.device)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    class_to_idx = ckpt.get("class_to_idx")
    print("Checkpoint chargé:", checkpoint_path)
    print("Backbone:", cfg.backbone_name, "| temporal_module:", cfg.temporal_module)
    print("clip_len:", cfg.clip_len, "| frame_size:", cfg.frame_size)
    return model, cfg, class_to_idx


def get_dataset_for_split(cfg: CFG, split: str, class_to_idx=None):
    return UCF101VideoFolderDataset(
        root_dir=split_dir(cfg, split),
        clip_len=cfg.clip_len,
        frame_size=cfg.frame_size,
        train=False,
        class_to_idx=class_to_idx,
    )


@torch.no_grad()
def predict_eval(model: nn.Module, video: torch.Tensor):
    """
    Prédiction propre en mode eval().
    Cette fonction sert de référence pour l'accuracy.
    On ne doit pas utiliser un forward Grad-CAM en mode train pour mesurer l'accuracy.
    """
    model.eval()
    logits = model(video, return_explanations=False)
    probs = torch.softmax(logits, dim=1)
    pred_idx = int(probs.argmax(dim=1).item())
    prob = float(probs[0, pred_idx].detach().cpu())
    return pred_idx, prob, logits


def temporal_importance_metrics(temporal_scores: torch.Tensor, clip_len: int) -> Dict[str, float]:
    """
    Métriques de l'axe temporel basées sur l'importance finale de l'explication.

    Contrairement à beta seul, ces scores proviennent de la heatmap finale
    Grad-CAM x porte spatiale x attention temporelle. Ils évaluent donc mieux
    l'axe "quand" réellement affiché.
    """
    s = temporal_scores.detach().float().view(-1)

    if s.numel() != clip_len:
        s = F.interpolate(
            s.view(1, 1, -1),
            size=clip_len,
            mode="linear",
            align_corners=False
        ).view(-1)

    s_np = s.cpu().numpy().astype(np.float64)
    s_np = np.clip(s_np, 0, None)
    p = s_np / (s_np.sum() + 1e-8)

    entropy = float(-(p * np.log(p + 1e-8)).sum() / np.log(len(p) + 1e-8))
    peak = int(np.argmax(p))
    concentration = float(p.max())

    sorted_p = np.sort(p)[::-1]
    top80_frames = int(np.searchsorted(np.cumsum(sorted_p), 0.80) + 1)

    threshold = np.percentile(p, 75)
    coverage = float((p >= threshold).mean())

    return {
        "temporal_explanation_entropy": entropy,
        "temporal_explanation_peak_frame": peak,
        "temporal_explanation_concentration": concentration,
        "temporal_explanation_top80_frames": top80_frames,
        "temporal_explanation_coverage_ratio": coverage,
    }


def temporal_gate_diagnostic_metrics(alpha: torch.Tensor, beta: torch.Tensor, clip_len: int) -> Dict[str, float]:
    """
    Diagnostic interne de la porte temporelle.
    Ces métriques ne sont pas utilisées comme métriques principales d'explication,
    mais elles permettent de vérifier si la porte LSTM-attention est sélective.
    """
    a = F.interpolate(alpha.detach().float().view(1, 1, -1), size=clip_len, mode="linear", align_corners=False).view(-1)
    b = F.interpolate(beta.detach().float().view(1, 1, -1), size=clip_len, mode="linear", align_corners=False).view(-1)

    a_np = np.clip(a.cpu().numpy().astype(np.float64), 0, None)
    a_np = a_np / (a_np.sum() + 1e-8)

    b_np = np.clip(b.cpu().numpy().astype(np.float64), 0, None)

    alpha_entropy = float(-(a_np * np.log(a_np + 1e-8)).sum() / np.log(len(a_np) + 1e-8))
    alpha_concentration = float(a_np.max())
    beta_mean = float(b_np.mean())
    beta_std = float(b_np.std())

    return {
        "temporal_gate_alpha_entropy_diagnostic": alpha_entropy,
        "temporal_gate_alpha_concentration_diagnostic": alpha_concentration,
        "temporal_gate_beta_mean_diagnostic": beta_mean,
        "temporal_gate_beta_std_diagnostic": beta_std,
    }


def flatten_dynamics_metrics(dynamics: Dict[str, Any]) -> Dict[str, Any]:
    """
    Transforme le dictionnaire dynamique en colonnes CSV simples.
    """
    return {
        "dynamic_segment": dynamics.get("segment"),
        "dynamic_motion_intensity_label": dynamics.get("motion_intensity_label"),
        "dynamic_motion_intensity_value": dynamics.get("motion_intensity_value"),
        "dynamic_localization_label": dynamics.get("localization_label"),
        "dynamic_localization_value": dynamics.get("localization_value"),
        "dynamic_variation_label": dynamics.get("variation_label"),
        "dynamic_variation_value": dynamics.get("variation_value"),
        "dynamic_continuity": dynamics.get("continuity"),
        "dynamic_direction": dynamics.get("direction"),
        "dynamic_peak_frame": dynamics.get("peak_frame"),
    }


def explain_one_sample(
    checkpoint_path: str,
    sample_index: int = 0,
    split: str = "test",
    output_dir: Optional[str] = None
):
    model, cfg, class_to_idx = load_model_from_checkpoint(checkpoint_path)
    dataset = get_dataset_for_split(cfg, split, class_to_idx)
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    output_dir = output_dir or cfg.output_dir
    os.makedirs(output_dir, exist_ok=True)

    sample = dataset[sample_index]
    video = sample["video"].unsqueeze(0).to(cfg.device)
    label = sample["label"].to(cfg.device)

    # Prédiction propre en mode eval.
    pred_idx, prob, _ = predict_eval(model, video)
    pred_text = idx_to_class.get(pred_idx, str(pred_idx))

    # Grad-CAM ciblé sur la classe prédite par le forward eval propre.
    gradcam = VideoGradCAM(model)
    heatmap, alpha, beta, _, _ = gradcam(video, class_idx=pred_idx)

    with torch.no_grad():
        temporal_flow, _, _ = compute_optical_flow_targets(
            video,
            out_hw=(cfg.frame_size, cfg.frame_size),
            flow_size=cfg.flow_size
        )

    temporal_scores = heatmap_temporal_importance(heatmap)

    dynamics = describe_dynamics_from_flow(
        temporal_scores,
        temporal_flow[0].detach().cpu(),
        heatmap=heatmap,
    )

    save_path = os.path.join(output_dir, f"explanation_{split}_{sample_index}_{pred_text}.mp4")
    save_explanation_video(sample["video"], heatmap, beta, pred_text, prob, dynamics, save_path)

    dynamic_save_path = os.path.join(output_dir, f"dynamic_explanation_{split}_{sample_index}_{pred_text}.mp4")
    dynamic_visual_report = save_dynamic_explanation_video(
        video_tensor=sample["video"],
        heatmap=heatmap,
        temporal_scores=temporal_scores,
        pred_text=pred_text,
        prob=prob,
        dynamics=dynamics,
        save_path=dynamic_save_path,
        fps=8,
        mode="motion_guided",
    )

    del_auc, ins_auc, del_curve, ins_curve = temporal_deletion_insertion_score(
        model=model,
        video=video,
        target_class=pred_idx,
        temporal_scores=temporal_scores,
        fill_strategy="mean"
    )

    st_del_auc, st_ins_auc, st_del_drop, st_ins_gain, st_del_curve, st_ins_curve = spatiotemporal_deletion_insertion_score(
        model=model,
        video=video,
        target_class=pred_idx,
        heatmap=heatmap,
        steps=cfg.eval_steps,
        max_top_ratio=cfg.eval_spatiotemporal_top_ratio,
        fill_strategy=cfg.eval_deletion_fill,
        dilation=cfg.eval_spatial_dilation,
    )

    stability = heatmap_temporal_stability(heatmap)
    loc_proxy = localization_motion_proxy(heatmap, sample["video"], cfg)
    temp_metrics = {
        **temporal_importance_metrics(temporal_scores, cfg.clip_len),
        **temporal_gate_diagnostic_metrics(alpha, beta, cfg.clip_len),
    }

    report = {
        "split": split,
        "sample_index": sample_index,
        "video_path": sample["path"],
        "ground_truth": sample["class_name"],
        "prediction": pred_text,
        "correct": bool(pred_text == sample["class_name"]),
        "probability": float(prob),
        "important_dynamics": dynamics,
        **flatten_dynamics_metrics(dynamics),
        **temp_metrics,
        "temporal_deletion_auc": float(del_auc),
        "temporal_insertion_auc": float(ins_auc),
        "spatiotemporal_deletion_auc": float(st_del_auc),
        "spatiotemporal_insertion_auc": float(st_ins_auc),
        "spatiotemporal_deletion_drop": float(st_del_drop),
        "spatiotemporal_insertion_gain": float(st_ins_gain),
        "temporal_stability": float(stability),
        "motion_localization_proxy": float(loc_proxy),
        "output_video": save_path,
        "dynamic_output_video": dynamic_save_path,
        "dynamic_dominant_motion_direction": dynamic_visual_report.get("dominant_motion_direction"),
        "dynamic_motion_gradcam_correlation": dynamic_visual_report.get("motion_gradcam_correlation"),
        "dynamic_useful_motion_ratio_percent": dynamic_visual_report.get("useful_motion_ratio_percent"),
        "dynamic_temporal_motion_alignment": dynamic_visual_report.get("temporal_motion_alignment"),
        "deletion_curve": del_curve,
        "insertion_curve": ins_curve,
        "spatiotemporal_deletion_curve": st_del_curve,
        "spatiotemporal_insertion_curve": st_ins_curve,
    }

    print(json.dumps({k: v for k, v in report.items() if k not in ["deletion_curve", "insertion_curve", "spatiotemporal_deletion_curve", "spatiotemporal_insertion_curve"]}, indent=2, ensure_ascii=False))
    return report, sample


def evaluate_full_split(
    checkpoint_path: str,
    split: str = "test",
    output_dir: Optional[str] = None,
    max_samples: Optional[int] = None,
    save_every: int = 50,
):
    """
    Évaluation corrigée.

    Correction principale :
    - L'accuracy est calculée avec un forward model.eval() propre.
    - Grad-CAM est calculé ensuite, ciblé sur la classe prédite, sans activer Dropout/BatchNorm.
    - Cela corrige la chute artificielle de l'accuracy observée dans l'ancienne fonction.
    """
    model, cfg, class_to_idx = load_model_from_checkpoint(checkpoint_path)
    dataset = get_dataset_for_split(cfg, split, class_to_idx)
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    output_dir = output_dir or os.path.join(cfg.output_dir, "full_split_evaluation")
    os.makedirs(output_dir, exist_ok=True)

    print("Split évalué:", split)
    print("Nombre de vidéos:", len(dataset))
    print("Dossier de sortie:", output_dir)

    gradcam = VideoGradCAM(model)
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    rows = []

    for i in tqdm(range(n), desc=f"Evaluation {split}"):
        try:
            sample = dataset[i]
            video = sample["video"].unsqueeze(0).to(cfg.device)
            label = sample["label"].to(cfg.device)

            # Forward eval propre pour l'accuracy.
            pred_idx, prob, _ = predict_eval(model, video)
            pred_text = idx_to_class.get(pred_idx, str(pred_idx))

            # Grad-CAM ciblé sur la prédiction propre.
            heatmap, alpha, beta, _, _ = gradcam(video, class_idx=pred_idx)

            with torch.no_grad():
                temporal_flow, _, _ = compute_optical_flow_targets(
                    video,
                    out_hw=(cfg.frame_size, cfg.frame_size),
                    flow_size=cfg.flow_size
                )

            temporal_scores = heatmap_temporal_importance(heatmap)

            dynamics = describe_dynamics_from_flow(
                temporal_scores,
                temporal_flow[0].detach().cpu(),
                heatmap=heatmap,
            )
            del_auc, ins_auc, del_curve, ins_curve = temporal_deletion_insertion_score(
                model=model,
                video=video,
                target_class=pred_idx,
                temporal_scores=temporal_scores,
                fill_strategy="mean"
            )

            st_del_auc, st_ins_auc, st_del_drop, st_ins_gain, st_del_curve, st_ins_curve = spatiotemporal_deletion_insertion_score(
                model=model,
                video=video,
                target_class=pred_idx,
                heatmap=heatmap,
                steps=cfg.eval_steps,
                max_top_ratio=cfg.eval_spatiotemporal_top_ratio,
                fill_strategy=cfg.eval_deletion_fill,
                dilation=cfg.eval_spatial_dilation,
            )

            stability = heatmap_temporal_stability(heatmap)
            loc_proxy = localization_motion_proxy(heatmap, sample["video"], cfg)
            temp_metrics = {
                **temporal_importance_metrics(temporal_scores, cfg.clip_len),
                **temporal_gate_diagnostic_metrics(alpha, beta, cfg.clip_len),
            }

            rows.append({
                "index": i,
                "video_path": sample["path"],
                "ground_truth": sample["class_name"],
                "prediction": pred_text,
                "correct": bool(pred_text == sample["class_name"]),
                "probability": float(prob),
                "important_dynamics": json.dumps(dynamics, ensure_ascii=False),
                **flatten_dynamics_metrics(dynamics),
                **temp_metrics,
                "temporal_deletion_auc": float(del_auc),
                "temporal_insertion_auc": float(ins_auc),
                "spatiotemporal_deletion_auc": float(st_del_auc),
                "spatiotemporal_insertion_auc": float(st_ins_auc),
                "spatiotemporal_deletion_drop": float(st_del_drop),
                "spatiotemporal_insertion_gain": float(st_ins_gain),
                "temporal_stability": float(stability),
                "motion_localization_proxy": float(loc_proxy),
                "deletion_curve": json.dumps(del_curve),
                "insertion_curve": json.dumps(ins_curve),
                "spatiotemporal_deletion_curve": json.dumps(st_del_curve),
                "spatiotemporal_insertion_curve": json.dumps(st_ins_curve),
            })

        except Exception as e:
            rows.append({"index": i, "error": str(e), "correct": False})

        if (i + 1) % save_every == 0:
            pd.DataFrame(rows).to_csv(os.path.join(output_dir, f"{split}_metrics_partial.csv"), index=False)

    df = pd.DataFrame(rows)
    csv_path = os.path.join(output_dir, f"{split}_metrics.csv")
    df.to_csv(csv_path, index=False)
    print("Métriques sauvegardées:", csv_path)
    return df


def summarize_results(df: pd.DataFrame, output_dir: Optional[str] = None):
    output_dir = output_dir or "./xai_outputs_attention_gated_final/full_split_evaluation"
    os.makedirs(output_dir, exist_ok=True)
    df_ok = df[df["error"].isna()].copy() if "error" in df.columns else df.copy()

    metrics = [
        "probability",
        "temporal_deletion_auc",
        "temporal_insertion_auc",
        "spatiotemporal_deletion_auc",
        "spatiotemporal_insertion_auc",
        "spatiotemporal_deletion_drop",
        "spatiotemporal_insertion_gain",
        "temporal_stability",
        "motion_localization_proxy",
        "temporal_explanation_entropy",
        "temporal_explanation_concentration",
        "temporal_explanation_top80_frames",
        "temporal_explanation_coverage_ratio",
        "temporal_gate_alpha_entropy_diagnostic",
        "temporal_gate_alpha_concentration_diagnostic",
        "temporal_gate_beta_mean_diagnostic",
        "temporal_gate_beta_std_diagnostic",
        "dynamic_motion_intensity_value",
        "dynamic_localization_value",
        "dynamic_variation_value",
    ]

    summary = {
        "num_samples_total": len(df),
        "num_samples_valid": len(df_ok),
        "accuracy": float(df_ok["correct"].mean()),
    }

    for m in metrics:
        if m in df_ok.columns:
            vals = pd.to_numeric(df_ok[m], errors="coerce")
            summary[f"{m}_mean"] = float(vals.mean())
            summary[f"{m}_std"] = float(vals.std())
            summary[f"{m}_median"] = float(vals.median())

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(os.path.join(output_dir, "summary_global_metrics.csv"), index=False)

    by_class = df_ok.groupby("ground_truth").agg(
        n=("ground_truth", "count"),
        accuracy=("correct", "mean"),
        confidence_mean=("probability", "mean"),
        temporal_deletion_auc_mean=("temporal_deletion_auc", "mean"),
        temporal_insertion_auc_mean=("temporal_insertion_auc", "mean"),
        spatiotemporal_deletion_auc_mean=("spatiotemporal_deletion_auc", "mean"),
        spatiotemporal_insertion_auc_mean=("spatiotemporal_insertion_auc", "mean"),
        spatiotemporal_deletion_drop_mean=("spatiotemporal_deletion_drop", "mean"),
        stability_mean=("temporal_stability", "mean"),
        motion_proxy_mean=("motion_localization_proxy", "mean"),
    ).reset_index().sort_values("accuracy")
    by_class.to_csv(os.path.join(output_dir, "summary_by_class.csv"), index=False)

    print("Diagnostic global")
    print("-----------------")
    print(f"Accuracy globale: {summary['accuracy']:.4f}")

    if "spatiotemporal_deletion_auc_mean" in summary:
        print(f"Deletion AUC spatio-temporelle: {summary['spatiotemporal_deletion_auc_mean']:.4f} | plus bas = meilleur")
    if "spatiotemporal_insertion_auc_mean" in summary:
        print(f"Insertion AUC spatio-temporelle: {summary['spatiotemporal_insertion_auc_mean']:.4f} | plus haut = meilleur")
    if "spatiotemporal_deletion_drop_mean" in summary:
        print(f"Deletion drop final: {summary['spatiotemporal_deletion_drop_mean']:.4f} | plus haut = plus causal")
    if "temporal_deletion_auc_mean" in summary:
        print(f"Deletion AUC temporelle frame-level: {summary['temporal_deletion_auc_mean']:.4f} | diagnostic secondaire")
    if "temporal_insertion_auc_mean" in summary:
        print(f"Insertion AUC temporelle frame-level: {summary['temporal_insertion_auc_mean']:.4f} | diagnostic secondaire")
    if "temporal_stability_mean" in summary:
        print(f"Stabilité moyenne: {summary['temporal_stability_mean']:.6f} | plus bas = meilleur")
    if "motion_localization_proxy_mean" in summary:
        print(f"Motion proxy moyen: {summary['motion_localization_proxy_mean']:.4f} | plus haut = meilleur")

    print("\nAxes complémentaires")
    print("--------------------")
    if "temporal_explanation_entropy_mean" in summary:
        print(f"Entropie explication temporelle: {summary['temporal_explanation_entropy_mean']:.4f} | plus bas = plus sélectif")
    if "temporal_explanation_concentration_mean" in summary:
        print(f"Concentration explication temporelle: {summary['temporal_explanation_concentration_mean']:.4f} | plus haut = plus sélectif")
    if "temporal_explanation_top80_frames_mean" in summary:
        print(f"Frames nécessaires pour 80% de l'importance: {summary['temporal_explanation_top80_frames_mean']:.2f} | plus bas = plus concentré")
    if "temporal_explanation_coverage_ratio_mean" in summary:
        print(f"Couverture temporelle explicative moyenne: {summary['temporal_explanation_coverage_ratio_mean']:.4f}")

    print("\nDiagnostic porte temporelle")
    print("---------------------------")
    if "temporal_gate_alpha_entropy_diagnostic_mean" in summary:
        print(f"Entropie alpha gate: {summary['temporal_gate_alpha_entropy_diagnostic_mean']:.4f}")
    if "temporal_gate_alpha_concentration_diagnostic_mean" in summary:
        print(f"Concentration alpha gate: {summary['temporal_gate_alpha_concentration_diagnostic_mean']:.4f}")
    if "temporal_gate_beta_mean_diagnostic_mean" in summary:
        print(f"Moyenne beta gate: {summary['temporal_gate_beta_mean_diagnostic_mean']:.4f}")
    if "temporal_gate_beta_std_diagnostic_mean" in summary:
        print(f"Écart-type beta gate: {summary['temporal_gate_beta_std_diagnostic_mean']:.4f}")

    print("\nAxe dynamique")
    print("-------------")
    if "dynamic_motion_intensity_value_mean" in summary:
        print(f"Intensité dynamique moyenne: {summary['dynamic_motion_intensity_value_mean']:.4f}")
    if "dynamic_localization_value_mean" in summary:
        print(f"Localisation dynamique moyenne: {summary['dynamic_localization_value_mean']:.2f}%")
    if "dynamic_variation_value_mean" in summary:
        print(f"Variation dynamique moyenne: {summary['dynamic_variation_value_mean']:.4f}")

    return summary_df, by_class


def export_outputs(output_dir="./xai_outputs"):
    zip_path = "/kaggle/working/xai_outputs_export"
    shutil.make_archive(zip_path, "zip", output_dir)
    print("Archive prête:", zip_path + ".zip")


## 10. Exécution expérimentale

Les cellules suivantes sont les points d'entrée pratiques du notebook.

Selon le besoin, on peut :

- entraîner un nouveau modèle ;
- reprendre un entraînement interrompu ;
- charger un checkpoint déjà entraîné ;
- générer une explication qualitative ;
- lancer l'évaluation complète.


In [12]:
# Entraînement d'un nouveau modèle

model, optimizer, cfg = train_main()

Train videos: 10055 | Val videos: 1673 | Test videos: 1723
Classes: 101
Downloading: "https://download.pytorch.org/models/s3d-d76dad2f.pth" to /root/.cache/torch/hub/checkpoints/s3d-d76dad2f.pth


100%|██████████| 32.0M/32.0M [00:00<00:00, 138MB/s] 
train:  20%|█▉        | 500/2514 [21:05<1:27:28,  2.61s/it]

Checkpoint sauvegardé: ./xai_outputs_attention_gated_final/batch_safety_checkpoint.pt


train:  40%|███▉      | 1000/2514 [42:21<1:07:09,  2.66s/it]

Checkpoint sauvegardé: ./xai_outputs_attention_gated_final/batch_safety_checkpoint.pt


...

Checkpoint sauvegardé: ./xai_outputs_attention_gated_final/batch_safety_checkpoint.pt


train:  40%|███▉      | 1000/2514 [44:19<1:09:47,  2.77s/it]

Checkpoint sauvegardé: ./xai_outputs_attention_gated_final/batch_safety_checkpoint.pt


train:  47%|████▋     | 1176/2514 [52:08<59:19,  2.66s/it]  


KeyboardInterrupt: 

In [ ]:
# Reprise d'un entraînement si un checkpoint est disponible

model, optimizer, cfg = train_main(
    resume_from="/kaggle/input/models/younessouarda/checkpt4/pytorch/default/1/last_checkpoint (19).pt"
)

In [13]:
# Chargement du meilleur checkpoint disponible

checkpoint_path = "/kaggle/input/models/younessouarda/att-gated-gradcam-model/pytorch/default/1/best_attention_gated_final_model (8).pt"
checkpoint_path

'/kaggle/input/models/younessouarda/att-gated-gradcam-model/pytorch/default/1/best_attention_gated_final_model (8).pt'

In [39]:
# Explication qualitative d'une vidéo test

report, sample = explain_one_sample(
    checkpoint_path=checkpoint_path,
    sample_index=40,
    split="test"
)

show_original_and_explanation(report, sample, width=750)

Checkpoint chargé: /kaggle/input/models/younessouarda/att-gated-gradcam-model/pytorch/default/1/best_attention_gated_final_model (8).pt
Backbone: s3d | temporal_module: lstm_attention
clip_len: 32 | frame_size: 224
{
  "split": "test",
  "sample_index": 40,
  "video_path": "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/Archery/v_Archery_g11_c03.avi",
  "ground_truth": "Archery",
  "prediction": "Archery",
  "correct": true,
  "probability": 0.9970883727073669,
  "important_dynamics": {
    "segment": "frames 16 à 23",
    "motion_intensity_label": "faible",
    "motion_intensity_value": 0.0342,
    "localization_label": "localisé",
    "localization_value": 75.0,
    "variation_label": "faible",
    "variation_value": 0.0079,
    "continuity": "stable",
    "direction": "mouvement stable",
    "peak_frame": 20
  },
  "dynamic_segment": "frames 16 à 23",
  "dynamic_motion_intensity_label": "faible",
  "dynamic_motion_intensity_value": 0.0342,
  "dynamic_localizatio

Vidéo expliquée — Grad-CAM spatio-temporel


Vidéo expliquée — Axe dynamique par Optical Flow


## 11. Évaluation sur le split Test

Cette cellule lance l'évaluation sur toutes les vidéos du split `test`.


In [21]:
df_test = evaluate_full_split(
    checkpoint_path=checkpoint_path,
    split="test",
    max_samples=None
)

summary_df, class_summary = summarize_results(df_test)

Checkpoint chargé: /kaggle/input/models/younessouarda/att-gated-gradcam-model/pytorch/default/1/best_attention_gated_final_model (8).pt
Backbone: s3d | temporal_module: lstm_attention
clip_len: 32 | frame_size: 224
Split évalué: test
Nombre de vidéos: 1723
Dossier de sortie: ./xai_outputs_attention_gated_final/full_split_evaluation


Evaluation test: 100%|██████████| 1723/1723 [1:06:08<00:00,  2.30s/it]


Métriques sauvegardées: ./xai_outputs_attention_gated_final/full_split_evaluation/test_metrics.csv
Diagnostic global
-----------------
Accuracy globale: 0.9048
Deletion AUC spatio-temporelle: 0.4876 | plus bas = meilleur
Insertion AUC spatio-temporelle: 0.7127 | plus haut = meilleur
Deletion drop final: 0.7100 | plus haut = plus causal
Deletion AUC temporelle frame-level: 0.7539 | diagnostic secondaire
Insertion AUC temporelle frame-level: 0.8137 | diagnostic secondaire
Stabilité moyenne: 0.002598 | plus bas = meilleur
Motion proxy moyen: 0.2715 | plus haut = meilleur

Axes complémentaires
--------------------
Entropie explication temporelle: 0.8834 | plus bas = plus sélectif
Concentration explication temporelle: 0.0706 | plus haut = plus sélectif
Frames nécessaires pour 80% de l'importance: 15.30 | plus bas = plus concentré
Couverture temporelle explicative moyenne: 0.2645

Diagnostic porte temporelle
---------------------------
Entropie alpha gate: 1.0000
Concentration alpha gate: 0.

##

# Conclusion du notebook

Ce notebook constitue la base expérimentale principale de l'approche Attention-Gated Grad-CAM.
 
Il produit les résultats nécessaires pour évaluer :

 - la performance de classification ;
 - la fidélité spatio-temporelle ;
 - la stabilité des explications ;
 - l'alignement dynamique avec le mouvement.
 